# Predicción con ML para un plan telefónico 

La compañía móvil Megaline tiene distintos planes de datos para celulares. Muchos de los usuarios geredaros el plan que actualmente tienen. Por lo que se pide desarrollar un modelo que pueda analizar el comportamiento de los clientes y recomendar uno de los nuevos planes de Megaline: Smart o Ultra.

Los datos con los que se cuenta son del comportamiento de los suscriptores que ya se han cambiado a los planes nuevos. Para esta tarea de clasificación se crear un modelo que escoja el plan correcto, con la mayor exactitud posible. En este proyecto, el umbral de exactitud es 0.75 y se utiliza el dataset para comprobar la exactitud.

Acontinuación se evaluan los modelos de clasificacion: DecisionTreeClassifier, RandomForestClassifier, LogisticRegression, para datos de una compañia movil y se determinar el plan a recomendar: Ultra o Smart. 

# Contenido <a id='back'></a>

* [1. Lectura de los datos](#data_review)
* [2. Segmentación de conjunto de datos](#seg)
* [3. Modelo con el algoritmo RandomForestClassifier](#random_forest)
* [4. Modelo con el algoritmo DesicionThreeClassifier](#desicion_three)
* [5. Modelo con el algoritmo LogasticRegression](#log_reg)
* [6. Seleccion de mejor modelo](#best_model)
* [Conclusiones](#end)

# 1. Lectura de los datos <a id='data_review'></a>

In [4]:
# Importar librerias a utilizar, asi como los modelos para clasificacion:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [5]:
#lectura de datos
df_megaline=pd.read_csv('users_behavior.csv')

In [6]:
#informacion para conocer columna objetivo 
df_megaline.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


Se observa que no hay datos nulos y el tipo de datos es correcto para cada columna.

# 2. Segmentación de conjunto de datos <a id='seg'></a>
La primera segmentación es 60% para datos de entrenamiento y 40% para datos de validación
La segunda segmentación se realiza ocupando el conjunto de datos de validación para segmentarlo en 2 partes, en partes iguales que serian, 20% para datos de validación y 20% para datos de test/prueba

In [7]:
# Si shuffle=False, el conjunto de datos no mezclará antes del particionado de esta forma evitamos que la 
#segmentacion del conjunto de datos sea de manera aleatoria cada vez que se ejecuta el script.

#primera segmentación
train_set_1, test_set_1 = train_test_split(df_megaline, test_size=0.4, random_state=42, shuffle=False)
#segunda segmentación
valid_set_1, test_set_1 = train_test_split(test_set_1, test_size=0.5, random_state=42, shuffle=False)


In [8]:
#comprobacion de particion de datos
print(train_set_1.shape)
print(test_set_1.shape)
print(df_megaline.shape)

(1928, 5)
(643, 5)
(3214, 5)


In [9]:
#eliminamos la columna objetivo "is_ultra" del dataframe completo
# y asignamos la columna  "is_ultra" a  target
def remove_labels(df, label_name):
    X = df.drop(label_name, axis=1)
    y = df[label_name].copy()
    return (X, y)

In [10]:
#se aplica la función creada para eliminar columna target a los conjuntos de entrenamiento, validacion y prueba.
features_train, target_train = remove_labels(train_set_1, 'is_ultra')
features_valid, target_valid = remove_labels(valid_set_1, 'is_ultra')
features_test, target_test = remove_labels(test_set_1, 'is_ultra')

# 3. Modelo con el algoritmo **RandomForestClassifier** <a id='random_forest'></a>
Entrenamiento del modelo de Bosques aleatorios con datos de entrenamiento y evaluación del modelo con datos de validación, se dejan los valores de prueba hasta determinar el mejor modelo
Se utiliza un rango de árboles de 3 a 20 y se evaluan cuántos árboles son necesarios para la mayor exactitud.


In [11]:
best_score = 0
best_est = 0
for est in range(3, 20): # selecciona el rango del hiperparámetro
    model = RandomForestClassifier(random_state=54321, n_estimators=est ) # configura el número de árboles
    model.fit(features_train, target_train) # entrena el modelo en el conjunto de entrenamiento

    # Evaluación del modelo
    score = model.score(features_valid, target_valid) # calcula la puntuación de accuracy en el conjunto de validación
    if score > best_score:
        best_score = score# guarda la mejor puntuación de accuracy en el conjunto de validación
        best_est = est # guarda el número de estimadores que corresponden a la mejor puntuación de exactitud

print("La exactitud del mejor modelo en el conjunto de validación (n_estimators = {}): {}".format(best_est, best_score))

La exactitud del mejor modelo en el conjunto de validación (n_estimators = 14): 0.8087091757387247


# 4. Modelo con el algoritmo **DecisionTreeClassifier** <a id='desicion_three'></a>
Entrenamiento del modelo de Árbol de desición con datos de entrenamiento y evaluación del modelo con datos de validación, se dejan los valores de prueba hasta determinar el mejor modelo.
Se utiliza un rango de profundidad de 1 a 11 y se evalua en cual profundidad es la mejor exactitud.


In [12]:
best_depth=0
best_score=0
for depth in range(1, 11):  # selecciona el rango del hiperparámetro
    model_deci_tree= DecisionTreeClassifier(random_state=54321, max_depth=depth)
    model_deci_tree.fit(features_train, target_train)
    # crea un modelo, especifica random_state=54321 y max_depth=depth 
    # entrena el modelo 
    train_predictions = model_deci_tree.predict(features_train) #predicciones obtenidas con el conjunto de entrenamiento
    valid_predictions = model_deci_tree.predict(features_valid) #predicciones con el conjunto de validadción
    
    print("Exactitud de max_depth igual a", depth) # imprime la profundidad evaluada
    as_train= accuracy_score(target_train, train_predictions) #cálculo de la exactitud para el entrenamiento
    print("Conjunto de entrenamiento:", as_train) #imprime el conjunto y la exactitud

    as_valid= accuracy_score(target_valid,  valid_predictions)  #calculo de la exactitud para la validación
    print("Conjunto de Validación:", as_valid) #imprime el conjunto y la exactitud
     
    print()
    if as_valid > best_score:
        best_score = as_valid# guarda la mejor puntuación de accuracy en el conjunto de validación
        best_depth = depth # guarda el número de estimadores que corresponden a la mejor puntuación de exactitud
    print("La exactitud del mejor modelo en el conjunto de validación (depth= {}): {}".format(best_depth, best_score))
    # imprime el mejor resultado de exactitud para el conjunto de validación

Exactitud de max_depth igual a 1
Conjunto de entrenamiento: 0.7494813278008299
Conjunto de Validación: 0.7511664074650077

La exactitud del mejor modelo en el conjunto de validación (depth= 1): 0.7511664074650077
Exactitud de max_depth igual a 2
Conjunto de entrenamiento: 0.783195020746888
Conjunto de Validación: 0.7822706065318819

La exactitud del mejor modelo en el conjunto de validación (depth= 2): 0.7822706065318819
Exactitud de max_depth igual a 3
Conjunto de entrenamiento: 0.7966804979253111
Conjunto de Validación: 0.7978227060653188

La exactitud del mejor modelo en el conjunto de validación (depth= 3): 0.7978227060653188
Exactitud de max_depth igual a 4
Conjunto de entrenamiento: 0.8034232365145229
Conjunto de Validación: 0.8040435458786936

La exactitud del mejor modelo en el conjunto de validación (depth= 4): 0.8040435458786936
Exactitud de max_depth igual a 5
Conjunto de entrenamiento: 0.8060165975103735
Conjunto de Validación: 0.807153965785381

La exactitud del mejor mode

# 5. Modelo con el algoritmo **LogisticRegression** <a id='log_reg'></a>
Entrenamiento del modelo de Regresión Logística con datos de entrenamiento y evaluación del modelo con datos de validación, se dejan los valores de prueba hasta determinar el mejor modelo.
Se utiliza solver='liblinear como hiperparámetro de regresión general para pocos datos y muchas características.


In [13]:
#Modelo de regresiín logística
modelo_reg_log = LogisticRegression(random_state=54321, solver='liblinear') # inicializa el constructor de regresión logística con los parámetros random_state=54321 y solver='liblinear'
modelo_reg_log.fit(features_train, target_train) # entrena el modelo en el conjunto de entrenamiento
score_train = model.score(features_train, target_train) # calcula la puntuación de accuracy en el conjunto de entrenamiento
score_valid = model.score(features_valid, target_valid) # calcula la puntuación de accuracy en el conjunto de validación

print("Accuracy del modelo de regresión logística en el conjunto de entrenamiento:", score_train) #imprime la exactitud para el conjunto de entrenamiento
print("Accuracy del modelo de regresión logística en el conjunto de validación:", score_valid) #imprime la exactitud para el conjunto de validación

Accuracy del modelo de regresión logística en el conjunto de entrenamiento: 0.9963692946058091
Accuracy del modelo de regresión logística en el conjunto de validación: 0.80248833592535


# 6. Selección de mejor modelo <a id='best_model'></a>
La mejor extactitud de entre los modelos evaluados fue de 0.8149 con **DecisionTreeClassifier** con una profundidad de 8, por lo cual el conjuto de prueba se evalua en este modelo. 


In [14]:
#Se evalua el modelo de Árbol de decisión con una profundidad de 8
model_final= DecisionTreeClassifier(random_state=54321, max_depth=8)
model_final.fit(features_train, target_train) #se entrena el modelo
test_predictions = model_final.predict(features_test) #se introduce el conjunto de prueba, que se aislo especificamente para esta prueba
as_test= accuracy_score(target_test, test_predictions) #se calcula el valor de la exactitud
print("Exactitud del conjunto de prueba del mejor modelo, arbol de desicion:", as_test) #imprime los resultados de exactitud de predición

Exactitud del conjunto de prueba del mejor modelo, arbol de desicion: 0.80248833592535


# Conclusiones <a id='end'></a>
Mediante modelos de Machine Learning se predijo el mejor plan telefónico para los clientes de Megaline. Esto le sirve a la empresa para promocionar el servicio y las ventajas a las clientes asi como proporcionar un mejor servicio al sugerir un mejor plan segun el consumo de cada cliente. 

Los modelos evaluados fueron, Árbol de Desición, Bosques Aleatorios y Regresión Logística. Los tres modelos obtubieron una exactitud mayor al 80%, sin embargo el modelo de árbol de decisión obtuvo la mayor exactitud con una profundidad de 8, al utilizar este modelo con los datos de prueba se obtuvo una exactud del 0.8024 superando el umbral objetivo de 0.75